# Munk Translation Pipeline
## *Guide des égarés* (Paris, 1856) — Part 1 → English

Fetches Munk's French from Sefaria's API, translates each segment via Claude, and saves a structured JSON file usable by the trifold reader.

**Run cells in order.** Progress is saved after every segment — if the notebook crashes or times out, re-run the **▶ Run Pipeline** cell and it will resume automatically from the last completed segment.

---

## Step 1 — Install dependencies
*(Run once per session)*

In [1]:
!pip install -q google-genai requests tqdm
print('Done.')

Done.


## Step 2 — Imports

In [2]:
from google import genai
import requests
import json, os, re, time, datetime
from getpass import getpass
from tqdm.notebook import tqdm
from IPython.display import display, HTML
print('Imports OK.')

Imports OK.


## Step 3 — Gemini API key
Get yours at [console.anthropic.com](https://console.anthropic.com). It will not be displayed after you paste it.

In [3]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)
print('Client initialized.')

Client initialized.


## Step 4 — Configuration
Edit the values in this cell if needed, then run it.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# ── Model ──────────────────────────────────────────────────────────────
# gemini-2.5-flash  →  fast, high quality, ~$4-8 for all of Part 1
# claude-opus-4-6    →  highest quality,    ~$20-30 for all of Part 1
MODEL = 'gemini-3-flash-preview'

# ── Output file ─────────────────────────────────────────────────────────
OUTPUT_FILE = '/content/drive/MyDrive/Munk''s Guide/munk_translations.json'

# ── Rate limiting ────────────────────────────────────────────────────────
# Seconds to wait between API calls. Increase to 2.0 if you hit errors.
DELAY = 3.0

# ── Sefaria version string for Munk's French ────────────────────────────
MUNK_VER = 'french|Guide_des_\u00e9gar\u00e9s,_trans._by_Salomon_Munk,_Paris,_1856_[fr]'
TEXT_BASE = 'Guide_for_the_Perplexed,_Part_1'

print(f'Model: {MODEL}')
print(f'Output: {OUTPUT_FILE}')

Model: gemini-3-flash-preview
Output: /content/drive/MyDrive/Munks Guide/munk_translations.json


## Step 5 — System prompt and glossary
These are pre-filled from your prompt design document. Expand the glossary here before starting a full run.

In [8]:
SYSTEM_PROMPT = '''ROLE AND TASK
You are translating the main body of Salomon Munk's French philosophical
translation of Maimonides' Guide for the Perplexed, published as Guide des
egarés, Paris, 1856. Munk's French is itself a translation from the original
Judeo-Arabic (Dalalat al-Ha'irin). Your sole task is to render Munk's French
into English faithfully and completely.

You are translating Munk — not Maimonides. Do not consult, harmonize with,
or import phrasing from other English translations of the Guide (Friedländer
1881, Pines 1963, or any other). If you are aware of how another translator
renders a given passage, set that knowledge aside entirely.

REGISTER
Adopt the register of serious Victorian scholarly prose: formal, precise,
somewhat elevated, but not artificially archaic. Prefer Latinate vocabulary
where Munk makes Latinate choices in French (e.g., "faculty" not "ability";
"intellect" not "mind"; "substance" not "stuff"; "apprehension" not "grasp").
Munk's sentences are often long and periodic; preserve their syntactic
structure where English permits, rather than breaking them into shorter units.
Do not modernize. Do not colloquialize.

COMPLETENESS
Translate every word Munk wrote. Do not summarize, compress, or omit any
portion of the input. Do not add explanatory glosses, parenthetical
clarifications, or interpretive expansions of your own.

MULTILINGUAL CONTENT — CRITICAL RULES
Munk frequently embeds Hebrew, Arabic, Greek, and Latin within his French
prose. Handle each as follows:

  Hebrew in Hebrew script: preserve exactly as Munk has it, including any
  vowel pointing (niqqud) he supplies. Do not transliterate what Munk has
  written in Hebrew characters. Do not normalize or modernize the spelling.

  Munk's own transliterations of Hebrew or Arabic into Latin characters
  (e.g., "selèm", "Kalâm", "Moré Neboukhim"): preserve his transliteration
  system exactly, including all diacritics, exactly as printed. Do not
  standardize to modern academic transliteration conventions.

  Arabic or Hebrew in their respective scripts: preserve exactly as Munk has them. Do not translate these snippets. Reproduce the Arabic or Hebrew characters in your response precisely as they appear in the source.

  Latin quotations or phrases: translate into English inline, in square
  brackets immediately following the Latin, marked [Lat.: ...].

  Greek terms or phrases: preserve in Greek script as Munk has them.

  Biblical and rabbinic citations: preserve Munk's citation form. Use the
  English name of the book (e.g., "Psalms" for "Psaumes") but do not alter
  his chapter/verse numbering.

MUNK'S FOOTNOTES
Munk's edition contains extensive translator's footnotes (notes de bas de
page). These footnotes are a critical part of his scholarly apparatus and
must be preserved in full. Do not omit, summarize, or compress them.

Render each footnote using standard Markdown footnote syntax:
  - At the point in the body text where Munk places his footnote marker,
    insert a reference anchor: [^1], [^2], etc., numbered sequentially
    within each segment starting from 1.
  - At the end of the translated segment, collect all footnote bodies
    in order, each on its own line, formatted as:
    [^1]: Translated text of Munk's first footnote.
    [^2]: Translated text of Munk's second footnote.

Translate the content of the footnotes with the same fidelity, register,
and rules that govern the main body text. The same glossary, multilingual
preservation rules, and completeness requirements apply inside footnotes.
If a footnote itself contains Hebrew, Arabic, Greek, or Latin, handle it
exactly as specified in the Multilingual Content rules above.

TRANSLATOR'S NOTES
You may add a translator's note only when:
  (a) a French word or phrase is genuinely ambiguous between two readings
      with meaningfully different philosophical implications, or
  (b) a proper name, technical term, or abbreviation is unresolvable.
Keep notes to one sentence. Do not add notes for routine difficulties.
Place these notes in the `tr_notes` list in your JSON output, NOT in the English text.

STRICT OUTPUT FORMAT
You will be given a JSON array of segments to translate. You MUST return a JSON array containing the exact same number of items.
For each item, provide:
- `ref`: the exact Sefaria reference provided in the input.
- `english`: your full English translation of the segment, including any inline footnotes and footnote bodies appended at the end.
- `tr_notes`: A list of strings. If a French word is ambiguous or a technical term needs clarification, provide a translator's note here (e.g. "faculty: Munk distinguishes..."). Otherwise leave empty.
'''

GLOSSARY = ''' TERMINOLOGY GLOSSARY — apply strictly throughout

CORE PHILOSOPHICAL TERMS
  intellect          → intellect          (not "mind")
  entendement        → understanding      (distinct from intellect; preserve distinction)
  forme              → form               (not "shape")
  matière            → matter             (not "material" or "stuff")
  faculté            → faculty            (not "ability" or "power")
  imagination        → imagination
  perfection         → perfection         (Aristotelian sense)
  agent              → agent
  cause efficiente   → efficient cause
  hypostase          → hypostasis         (not "substance")
  attributs          → attributes         (theological sense)
  essence            → essence
  accident           → accident           (Aristotelian category)
  substance          → substance          (Aristotelian category)
  mouvement          → motion             (not "movement"; Aristotelian)
  repos              → rest               (counterpart to mouvement)
  âme                → soul              (not "mind")
  puissance          → potentiality       (Aristotle's dunamis)
  acte               → actuality          (Aristotle's energeia)
  matière première   → prime matter
  intelligence séparée → separate intellect

MUNK'S TRANSLITERATIONS — preserve exactly, including diacritics
  Kalâm, Motécallemîn, Moré Neboukhim, selèm, El, Adonaï

PROPER NAMES — Anglicize consistently
  Aristote → Aristotle | Platon → Plato | Averroès → Averroes
  Avicenne → Avicenna | Maïmonide → Maimonides
  Ibn Tibbon → Ibn Tibbon | Ibn Ézra → Ibn Ezra

[Expand this glossary before starting the full run — see the companion .md file]'''


## Step 6 — Sefaria API utilities
Probes the Sefaria API chapter by chapter to build the complete list of segments and fetch their French text.

In [ ]:
import json

LOCAL_JSON_PATH = '/content/drive/MyDrive/Munks Guide/French.json'

def flatten_text(t):
    '''Sefaria sometimes returns nested lists; flatten to plain string.'''
    if isinstance(t, list):
        return ' '.join(flatten_text(x) for x in t)
    return t or ''

def build_segment_list():
    '''
    Loads the ENTIRE book structure from local Sefaria JSON export.
    Returns a list of dicts: {ref, chapter, paragraph, french}.
    '''
    print('Loading text from local Sefaria JSON export...')
    with open(LOCAL_JSON_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    segments = []

    # 1. Letter to R Joseph
    letter = data['text'].get('Letter to R Joseph son of Judah', [])
    for i, t in enumerate(letter):
        french = flatten_text(t).strip()
        if french:
            segments.append({
                'ref': f'Guide_for_the_Perplexed,_Letter_to_R_Joseph_son_of_Judah.{i+1}',
                'chapter': 'Letter',
                'paragraph': i + 1,
                'french': french
            })

    # 2. Prefatory Remarks
    prefatory = data['text'].get('Prefatory Remarks', [])
    for i, t in enumerate(prefatory):
        french = flatten_text(t).strip()
        if french:
            segments.append({
                'ref': f'Guide_for_the_Perplexed,_Prefatory_Remarks.{i+1}',
                'chapter': 'Prefatory',
                'paragraph': i + 1,
                'french': french
            })

    # 3. Parts 1, 2, 3
    for part in ['Part 1', 'Part 2', 'Part 3']:
        part_data = data['text'].get(part, {})

        # Introduction for the Part
        for i, t in enumerate(part_data.get('Introduction', [])):
            french = flatten_text(t).strip()
            if french:
                segments.append({
                    'ref': f'Guide_for_the_Perplexed,_{part.replace(" ", "_")},_Introduction.{i+1}',
                    'chapter': f'{part} Intro',
                    'paragraph': i + 1,
                    'french': french
                })

        # Chapters for the Part
        chapters = part_data.get('', [])
        for ch_idx, chapter_paras in enumerate(chapters):
            ch = ch_idx + 1
            for i, t in enumerate(chapter_paras):
                french = flatten_text(t).strip()
                if french:
                    segments.append({
                        'ref': f'Guide_for_the_Perplexed,_{part.replace(" ", "_")}.{ch}.{i+1}',
                        'chapter': f'{part} Ch {ch}',
                        'paragraph': i + 1,
                        'french': french
                    })

    print(f'\nTotal: {len(segments)} segments extracted.')
    return segments
# Run it
ALL_SEGMENTS = build_segment_list()
# Build a quick-lookup dict: ref → french text
FRENCH_BY_REF = {s['ref']: s['french'] for s in ALL_SEGMENTS}
SEG_REFS = [s['ref'] for s in ALL_SEGMENTS]  # ordered list of all refs


Loading text from local Sefaria JSON export...

Total: 1454 segments extracted.


In [ ]:
# --- OCR Enrichment Step ---
# Run this ONLY if you want to replace images in French.json with Arabic/Hebrew text.
import json, base64, re, time
from google.genai import types
from tqdm.notebook import tqdm

def run_ocr_enrichment():
    INPUT = '/content/drive/MyDrive/Munks Guide/French.json'
    OUTPUT = '/content/drive/MyDrive/Munks Guide/French_Arabic_Enriched.json'
    
    with open(INPUT, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Extract all unique images to save on API calls
    def find_all_images(node, images_set):
        if isinstance(node, dict): 
            for v in node.values(): find_all_images(v, images_set)
        elif isinstance(node, list): 
            for x in node: find_all_images(x, images_set)
        elif isinstance(node, str):
            for b64 in re.findall(r'data:image/[^;]+;base64,([a-zA-Z0-9+/=]+)', node):
                images_set.add(b64)
    
    unique_b64s = set()
    find_all_images(data, unique_b64s)
    print(f"Found {len(unique_b64s)} unique images to process.")
    
    ocr_map = {}
    pbar = tqdm(unique_b64s, desc="OCRing unique images")
    for b64 in pbar:
        try:
            img = types.Part.from_bytes(data=base64.b64decode(b64), mime_type='image/jpeg')
            # Using gemini-3-flash-preview as requested
            res = client.models.generate_content(
                model='gemini-3-flash-preview', 
                contents=['Provide ONLY the Arabic or Hebrew script of the word in this image. No other text.', img]
            )
            ocr_map[b64] = res.text.strip()
            time.sleep(0.2) # Small delay for safety
        except Exception as e:
            print(f"Error: {e}")
            ocr_map[b64] = "[OCR_ERROR]"

    # Replace in the data
    def replace_images(node):
        if isinstance(node, dict): return {k: replace_images(v) for k, v in node.items()}
        if isinstance(node, list): return [replace_images(x) for x in node]
        if isinstance(node, str):
            matches = re.findall(r'data:image/[^;]+;base64,([a-zA-Z0-9+/=]+)', node)
            for b64 in matches:
                word = ocr_map.get(b64, "[IMAGE]")
                # Replace the full img tag or just the source
                # We use a broad pattern to catch the whole tag
                tag_pattern = r'<img [^>]*data:image/[^;]+;base64,' + re.escape(b64) + r'[^>]*>' 
                node = re.sub(tag_pattern, f'<span dir="rtl">{word}</span>', node)
            return node
        return node

    enriched = replace_images(data)
    with open(OUTPUT, 'w', encoding='utf-8') as f: json.dump(enriched, f, ensure_ascii=False, indent=2)
    print(f"Done! Created {OUTPUT}")

run_ocr_enrichment()


## Step 7 — Load or initialise results file
If `OUTPUT_FILE` already exists (from a previous run), existing translations are loaded and will be skipped during the pipeline.

In [11]:
def load_results():
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, encoding='utf-8') as f:
            existing = json.load(f)
        done = len(existing.get('segments', {}))
        print(f'Resuming: {done} segments already translated.')
        return existing
    print('No existing file found — starting fresh.')
    return {
        'metadata': {
            'title': 'Guide des égarés — Part 1',
            'munk_edition': 'Paris, 1856',
            'model': MODEL,
            'part': 1,
            'created': datetime.datetime.now().isoformat(),
            'completed': False
        },
        'segments': {},
        'stats': {
            'total_segments': len(ALL_SEGMENTS),
            'completed': 0,
            'total_tokens': 0
        }
    }

def save_results(results):
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

RESULTS = load_results()

Resuming: 55 segments already translated.


## Step 8 — Translation engine

In [13]:
from pydantic import BaseModel
import json
import re

class TranslatedSegment(BaseModel):
    ref: str
    english: str
    tr_notes: list[str]

def translate_chunk(chunk_todo, prev_french='', **kwargs):
    '''
    Send a chunk of segments to Gemini and return a list of result dicts.
    Raises on API error (caller handles retry logic).
    '''
    payload = [{'ref': s['ref'], 'french': s['french']} for s in chunk_todo]
    
    context_block = (
        prev_french if prev_french
        else '[This is the first segment — no preceding context.]'
    )
    
    user_msg = f"CONTEXT — preceding segment (do not translate; for continuity only):\n\n{context_block}\n\nTRANSLATE THE FOLLOWING SEGMENTS:\n{json.dumps(payload, ensure_ascii=False, indent=2)}"

    # Combine SYSTEM_PROMPT and GLOSSARY for the system instruction
    full_system_instruction = f"{SYSTEM_PROMPT}\n\n{GLOSSARY}"

    response = client.models.generate_content(
        model=MODEL,
        contents=user_msg,
        config=genai.types.GenerateContentConfig(
            system_instruction=full_system_instruction,
            temperature=kwargs.get('temperature', 0.1),
            response_mime_type='application/json',
            response_schema=list[TranslatedSegment],
            max_output_tokens=8192
        )
    )
    
    raw_text = response.text
    # Fix invalid \u escapes (where \u is not followed by 4 hex digits)
    raw_text = re.sub(r'\\u(?![0-9a-fA-F]{4})', r'\\\\u', raw_text)
    
    if raw_text.startswith("```"):
        raw_text = re.sub(r"^```[a-z]*\n", "", raw_text)
        raw_text = re.sub(r"\n```$", "", raw_text)
        
    # Robust JSON cleanup for common Gemini truncation/malformation
    raw_text = raw_text.strip()
    if not raw_text.endswith(']') and not raw_text.endswith('}'):
        if raw_text.count('"') % 2 != 0:
            raw_text += '"'
        if raw_text.count('{') > raw_text.count('}'):
            raw_text += '}' * (raw_text.count('{') - raw_text.count('}'))
        if raw_text.count('[') > raw_text.count(']'):
            raw_text += ']' * (raw_text.count('[') - raw_text.count(']'))
        
    try:
        parsed_array = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON ERROR at char {e.pos}: {e.msg}")
        start = max(0, e.pos - 50)
        end = min(len(raw_text), e.pos + 50)
        print("Context around error:", raw_text[start:end])
        raise

    # --- VALIDATION ---
    if not isinstance(parsed_array, list):
        raise ValueError(f"Expected JSON list, got {type(parsed_array)}")
    
    if len(parsed_array) != len(chunk_todo):
        raise ValueError(f"Segment count mismatch: expected {len(chunk_todo)}, got {len(parsed_array)}")
    
    for i, item in enumerate(parsed_array):
        if not isinstance(item, dict):
            raise ValueError(f"Segment {i} is not a dict")
        if 'ref' not in item or 'english' not in item:
            raise KeyError(f"Missing required keys ('ref' or 'english') in segment {i}")
    # ------------------

    in_tok = response.usage_metadata.prompt_token_count if response.usage_metadata else 0
    out_tok = response.usage_metadata.candidates_token_count if response.usage_metadata else 0
    
    results = []
    for item in parsed_array:
        results.append({
            'ref': item['ref'],
            'english': item['english'],
            'tr_notes': item.get('tr_notes', []),
            'input_tokens': in_tok,
            'output_tokens': out_tok,
            'timestamp': datetime.datetime.now().isoformat()
        })
    return results

print('Translation chunk function ready.')


Translation function ready.


## Step 9 — Test mode *(run this before the full pipeline)*
Translates a 5-chapter chunk (e.g. Part 1 Intro + Chapters 1-4) so you can verify the output quality and check that the API key, Sefaria fetch, and prompts are all working correctly. It also ensures the request size works within Colab timeouts. Does **not** write to the output file.

Review the output carefully:
- Is the register right (formal, Latinate, Victorian scholarly)?
- Are Hebrew characters preserved exactly?
- Are Munk's transliterations (circumflexes, grave accents) intact?
- Are the glossary terms applied correctly?



In [14]:
def group_segments(segments):
    chunks = []
    # Batch 0: Letter & Prefatory
    batch_0 = [s for s in segments if s['chapter'] in ['Letter', 'Prefatory']]
    if batch_0: chunks.append(batch_0)
    
    # Batch 1: Part 1 Intro + Chapters 1-4
    batch_1 = [s for s in segments if s['chapter'] in ['Part 1 Intro', 'Part 1 Ch 1', 'Part 1 Ch 2', 'Part 1 Ch 3', 'Part 1 Ch 4']]
    if batch_1: chunks.append(batch_1)
    
    # Get the rest of the segments (Chapters 5 to 76)
    handled = ['Letter', 'Prefatory', 'Part 1 Intro', 'Part 1 Ch 1', 'Part 1 Ch 2', 'Part 1 Ch 3', 'Part 1 Ch 4']
    rest = [s for s in segments if s['chapter'] not in handled]
    
    chapters = []
    for s in rest:
        if s['chapter'] not in chapters:
            chapters.append(s['chapter'])
            
    # Chunk remaining by 1 (per chapter) for reliability
    for chap in chapters:
        chunk_segs = [s for s in rest if s['chapter'] == chap]
        chunks.append(chunk_segs)
        
    return chunks

CHUNKS = group_segments(ALL_SEGMENTS)

def run_test():
    print('=== TEST MODE: first 5-chapter chunk ===\n')
    divider = '─' * 65
    
    # Load what's already done so we test the exact first chunk we need
    try:
        temp_results = load_results()
        done_refs = set(temp_results['segments'].keys())
    except:
        done_refs = set()
        
    test_chunk = None
    chunk_todo = None
    
    for chunk in CHUNKS:
        ctodo = [s for s in chunk if s['ref'] not in done_refs]
        if ctodo:
            test_chunk = chunk
            chunk_todo = ctodo
            break
            
    if not test_chunk:
        print("All segments translated!")
        return

    # Find the immediately preceding segment for context
    first_todo_ref = chunk_todo[0]['ref']
    idx = SEG_REFS.index(first_todo_ref)
    prev_french = FRENCH_BY_REF.get(SEG_REFS[idx - 1], '') if idx > 0 else ''

    print(f'Translating {len(chunk_todo)} remaining segments from chapters: {list(dict.fromkeys(s["chapter"] for s in test_chunk))}...')
    print(f'(Skipping {len(test_chunk) - len(chunk_todo)} already translated segments)')
    
    start_time = time.time()
    results = translate_chunk(chunk_todo, prev_french)
    elapsed = time.time() - start_time
    
    print(f'Completed in {elapsed:.1f} seconds.\n')
    
    for i, seg in enumerate(chunk_todo[:3]): # just show first 3 for review
        result = next((r for r in results if r['ref'] == seg['ref']), None)
        if not result:
            print(f"Error: {seg['ref']} not in results!")
            continue
            
        print(divider)
        print(f'SEGMENT: {seg["ref"]}')
        print(f'FRENCH (first 300 chars):\n  {seg["french"][:300]}')
        print(f'ENGLISH (first 300 chars):\n  {result["english"][:300]}')
        if result['tr_notes']:
            print(f'TRANSLATOR NOTES: {result["tr_notes"]}')
        print(divider + '\n')
        
    if results:
        print(f'Tokens: {results[0]["input_tokens"]} in / {results[0]["output_tokens"]} out for the whole chunk.')
    print('Test complete. Proceed to the pipeline cell if output looks correct and timing is acceptable.')

run_test()


=== TEST MODE: first 3 segments ===

Translating Guide_for_the_Perplexed,_Letter_to_R_Joseph_son_of_Judah.1...
─────────────────────────────────────────────────────────────────
SEGMENT: Guide_for_the_Perplexed,_Letter_to_R_Joseph_son_of_Judah.1
FRENCH (first 300 chars):
  <b>AU NOM DE L'ÉTERNEL</b> <br>DIEU DE L'UNIVERS<sup class="footnote-marker">(2)</sup><i class="footnote">Nous avons traduit ici les mots <span dir="rtl">אל עולם</span> dans le sens que Maïmonide lui-même leur prête dans plusieurs endroits, et notamment dans le chap. 29 de la troisième partie du <i>
ENGLISH (first 300 chars):
  **IN THE NAME OF THE ETERNAL**
GOD OF THE UNIVERSE[^1]

[^1]: We have here translated the words אל עולם in the sense which Maimonides himself attributes to them in several places, and notably in chapter 29 of the third part of the *Guide*, although in the biblical passage (Genesis, 21, 33) these wor
Tokens: 1651 in / 89 out
─────────────────────────────────────────────────────────────────

Tran

In [ ]:
from pydantic import BaseModel
import json
import re

class TranslatedSegment(BaseModel):
    ref: str
    english: str
    tr_notes: list[str]

def translate_chunk(chunk_todo, prev_french='', **kwargs):
    '''
    Send a chunk of segments to Gemini and return a list of result dicts.
    Raises on API error (caller handles retry logic).
    '''
    payload = [{'ref': s['ref'], 'french': s['french']} for s in chunk_todo]
    
    context_block = (
        prev_french if prev_french
        else '[This is the first segment — no preceding context.]'
    )
    
    user_msg = f"CONTEXT — preceding segment (do not translate; for continuity only):\n\n{context_block}\n\nTRANSLATE THE FOLLOWING SEGMENTS:\n{json.dumps(payload, ensure_ascii=False, indent=2)}"

    # Combine SYSTEM_PROMPT and GLOSSARY for the system instruction
    full_system_instruction = f"{SYSTEM_PROMPT}\n\n{GLOSSARY}"

    response = client.models.generate_content(
        model=MODEL,
        contents=user_msg,
        config=genai.types.GenerateContentConfig(
            system_instruction=full_system_instruction,
            temperature=kwargs.get('temperature', 0.1),
            response_mime_type='application/json',
            response_schema=list[TranslatedSegment],
            max_output_tokens=8192
        )
    )
    
    raw_text = response.text
    # Fix invalid \u escapes (where \u is not followed by 4 hex digits)
    raw_text = re.sub(r'\\u(?![0-9a-fA-F]{4})', r'\\\\u', raw_text)
    
    if raw_text.startswith("```"):
        raw_text = re.sub(r"^```[a-z]*\n", "", raw_text)
        raw_text = re.sub(r"\n```$", "", raw_text)
        
    # Robust JSON cleanup for common Gemini truncation/malformation
    raw_text = raw_text.strip()
    if not raw_text.endswith(']') and not raw_text.endswith('}'):
        if raw_text.count('"') % 2 != 0:
            raw_text += '"'
        if raw_text.count('{') > raw_text.count('}'):
            raw_text += '}' * (raw_text.count('{') - raw_text.count('}'))
        if raw_text.count('[') > raw_text.count(']'):
            raw_text += ']' * (raw_text.count('[') - raw_text.count(']'))
        
    try:
        parsed_array = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON ERROR at char {e.pos}: {e.msg}")
        start = max(0, e.pos - 50)
        end = min(len(raw_text), e.pos + 50)
        print("Context around error:", raw_text[start:end])
        raise

    # --- VALIDATION ---
    if not isinstance(parsed_array, list):
        raise ValueError(f"Expected JSON list, got {type(parsed_array)}")
    
    if len(parsed_array) != len(chunk_todo):
        raise ValueError(f"Segment count mismatch: expected {len(chunk_todo)}, got {len(parsed_array)}")
    
    for i, item in enumerate(parsed_array):
        if not isinstance(item, dict):
            raise ValueError(f"Segment {i} is not a dict")
        if 'ref' not in item or 'english' not in item:
            raise KeyError(f"Missing required keys ('ref' or 'english') in segment {i}")
    # ------------------

    in_tok = response.usage_metadata.prompt_token_count if response.usage_metadata else 0
    out_tok = response.usage_metadata.candidates_token_count if response.usage_metadata else 0
    
    results = []
    for item in parsed_array:
        results.append({
            'ref': item['ref'],
            'english': item['english'],
            'tr_notes': item.get('tr_notes', []),
            'input_tokens': in_tok,
            'output_tokens': out_tok,
            'timestamp': datetime.datetime.now().isoformat()
        })
    return results

print('Translation chunk function ready.')


Translation function ready.


In [15]:
import json
import os

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    completed_refs = sorted(data.get('segments', {}).keys())
    if completed_refs:
        print(f"First translated segment in file: {completed_refs[0]}")
        print(f"Total segments translated so far: {len(completed_refs)}")
    else:
        print("The file exists but contains no translated segments yet.")
else:
    print(f"The file '{OUTPUT_FILE}' has not been created yet. Run the pipeline to start translating.")

First translated segment in file: Guide_for_the_Perplexed,_Letter_to_R_Joseph_son_of_Judah.1
Total segments translated so far: 55


## Step 10 — ▶ Run the full pipeline
Translates all segments, skipping any already in the output file. Safe to interrupt and re-run — progress is saved after every segment.

**Estimated time:** ~15–40 minutes for all of Part 1, depending on model and segment count.

In [16]:
import concurrent.futures

def run_pipeline(max_workers=3):
    global RESULTS
    RESULTS = load_results()
    done_refs = set(RESULTS['segments'].keys())
    
    todo_chunks = []
    for chunk in CHUNKS:
        chunk_todo = [s for s in chunk if s['ref'] not in done_refs]
        if chunk_todo:
            # Pre-calculate prev_french for this chunk while we are still in order
            first_todo_ref = chunk_todo[0]['ref']
            idx = SEG_REFS.index(first_todo_ref)
            prev_f = FRENCH_BY_REF.get(SEG_REFS[idx - 1], '') if idx > 0 else ''
            todo_chunks.append((chunk, chunk_todo, prev_f))

    if not todo_chunks:
        print('All segments already translated.')
        return

    print(f'Chapters to translate: {len(todo_chunks)}')
    print(f'Parallel workers     : {max_workers}\n')

    def translate_worker(item):
        chunk, chunk_todo, prev_french = item
        chaps = list(dict.fromkeys(s["chapter"] for s in chunk))
        
        for attempt in range(4):
            try:
                temp = 0.1 + (attempt * 0.1)
                results = translate_chunk(chunk_todo, prev_french, temperature=temp)
                return (chaps, results)
            except Exception as e:
                if '429' in str(e):
                    time.sleep(30 * (attempt + 1))
                elif attempt == 3:
                    print(f"Fatal error in {chaps}: {e}")
                    return (chaps, None)
        return (chaps, None)

    pbar = tqdm(total=len(todo_chunks), desc="Translating chapters")
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_chunk = {executor.submit(translate_worker, item): item for item in todo_chunks}
        
        for future in concurrent.futures.as_completed(future_to_chunk):
            chaps, chunk_results = future.result()
            if chunk_results:
                # Save results safely (main thread)
                for result in chunk_results:
                    ref = result['ref']
                    # Find original segment for chapter/para info
                    orig_seg = next(s for s in ALL_SEGMENTS if s['ref'] == ref)
                    RESULTS['segments'][ref] = {
                        'ref': ref, 'chapter': orig_seg['chapter'], 
                        'paragraph': orig_seg['paragraph'], 'french': orig_seg['french'],
                        'english': result['english'], 'tr_notes': result['tr_notes'],
                        'input_tokens': chunk_results[0]['input_tokens'],
                        'output_tokens': chunk_results[0]['output_tokens'],
                        'timestamp': result['timestamp']
                    }
                RESULTS['stats']['completed'] = len(RESULTS['segments'])
                save_results(RESULTS)
            pbar.update(1)

    RESULTS['metadata']['completed'] = True
    RESULTS['metadata']['finished'] = datetime.datetime.now().isoformat()
    save_results(RESULTS)
    print(f'\n✓ Complete. Total segments: {len(RESULTS["segments"])}')



In [1]:
# Ensure the directory exists before running
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

run_pipeline()

NameError: name 'os' is not defined

## Step 11 — Review progress and sample output
Run this at any point to check how far the pipeline has gotten and inspect a random segment.

In [ ]:
import random

def show_stats():
    if not os.path.exists(OUTPUT_FILE):
        print('No output file yet — run the pipeline first.')
        return
    with open(OUTPUT_FILE, encoding='utf-8') as f:
        data = json.load(f)
    stats = data['stats']
    segs = data['segments']
    meta = data['metadata']
    print(f'Progress   : {stats["completed"]}/{stats["total_segments"]} segments')
    pct = 100 * stats['completed'] / max(stats['total_segments'], 1)
    bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
    print(f'           : [{bar}] {pct:.1f}%')
    print(f'Tokens     : {stats["total_tokens"]:,}')
    est = stats['total_tokens'] * 0.075 / 1_000_000
    print(f'Est. cost  : ~${est:.2f} (Flash pricing)')
    print(f'Model      : {meta["model"]}')
    print(f'Completed  : {meta.get("completed", False)}')
    if segs:
        notes_count = sum(len(s.get('tr_notes', [])) for s in segs.values())
        print(f'Tr. notes  : {notes_count} logged')
        print()
        sample_ref = random.choice(list(segs.keys()))
        sample = segs[sample_ref]
        print(f'─── Random sample: {sample_ref} ───')
        print(f'FRENCH  : {sample["french"][:250]}...')
        print(f'ENGLISH : {sample["english"][:250]}...')
        if sample.get('tr_notes'):
            print(f'TR.NOTES: {sample["tr_notes"]}')

show_stats()

## Step 12 — Export translator's notes
Collects all `[tr. note: ...]` flags from the translation into a separate file for review. Check these before treating the translation as complete.

In [ ]:
def export_notes():
    if not os.path.exists(OUTPUT_FILE):
        print('No output file yet.')
        return
    with open(OUTPUT_FILE, encoding='utf-8') as f:
        data = json.load(f)
    notes_file = OUTPUT_FILE.replace('.json', '_translator_notes.json')
    notes = {}
    for ref, seg in data['segments'].items():
        if seg.get('tr_notes'):
            notes[ref] = {
                'chapter': seg['chapter'],
                'paragraph': seg['paragraph'],
                'notes': seg['tr_notes'],
                'french_context': seg['french'][:200],
                'english_context': seg['english'][:200]
            }
    with open(notes_file, 'w', encoding='utf-8') as f:
        json.dump(notes, f, ensure_ascii=False, indent=2)
    print(f'Exported {len(notes)} notes to {notes_file}')
    return notes_file

export_notes()

## Step 13 — Download output files
Downloads both the translation JSON and the translator's notes file to your local machine.

In [ ]:
from google.colab import files as colab_files

notes_file = OUTPUT_FILE.replace('.json', '_translator_notes.json')

if os.path.exists(OUTPUT_FILE):
    print(f'Downloading {OUTPUT_FILE}...')
    colab_files.download(OUTPUT_FILE)
else:
    print(f'Main output file not found: {OUTPUT_FILE}')

if os.path.exists(notes_file):
    print(f'Downloading {notes_file}...')
    colab_files.download(notes_file)
else:
    print('No translator notes file yet — run Step 12 first.')